# 04 – Shipping Cost Model
Orders are small parcels (avg. ~4.5 units, ~$16.5), so they are assumed to be shipped by an external parcel carrier rather than by the company's own transport.
Shipping cost is estimated with our own conversion formula of weight and distance to price, calibrated on three real carrier rate tables (see price variants below).

The current model is compared with two alternatives:
- **realistic** – orders shipped from the nearest factory that already makes products of the same division
- **optimum** – orders shipped from the nearest of all five factories (theoretical upper bound)

**Assumptions:**
- Weight: 2 oz per unit + 0.5 lb packaging per parcel; billable weight rounded up to the next full pound (carrier rule)
- Distance for pricing: straight-line miles (carrier zones are radius-based); each zone represented by the midpoint of its mileage band
- Factory assignment in the alternative scenarios: nearest factory by road distance (OpenRouteService, driving-hgv profile)
- Three price variants: **low** = USPS Ground Advantage commercial, **base** = USPS Ground Advantage retail (both Notice 123, effective July 12, 2026), **high** = UPS Ground published daily rates (effective Dec 22, 2025)
- Base rates only: fuel, residential and delivery-area surcharges are excluded

**How to read the results:**
- **Dollar costs are illustrative** – they come from our formula, not exact carrier prices. The reliable results are the **proportions (percent savings) and the miles saved**.
- To calculate the actual savings, the company's real shipping costs would be needed – the model can be re-run with them.

## Price formulas
Each rate table is fitted with: price = a + b·weight + c·miles + e·weight·miles

In [1]:
import numpy as np
import pandas as pd

# Midpoints of zone mileage bands, zones 1–8 (zone 8 = 1801+ miles → 2100)
ZONE_MILES = [25, 100, 225, 450, 800, 1200, 1600, 2100]

RATE_CARDS = {
    # USPS Ground Advantage commercial, Notice 123, effective July 12, 2026 (zones 1–8)
    "low_usps_commercial": {
        "zones": ZONE_MILES,
        "rates": {
            1: [7.61, 7.68, 8.00, 8.15, 8.74, 9.63, 9.98, 10.67],
            2: [7.99, 8.08, 8.26, 8.51, 9.95, 11.58, 12.00, 12.87],
            3: [8.64, 8.66, 9.14, 9.67, 11.57, 13.59, 14.36, 15.75],
            4: [9.28, 9.34, 9.70, 10.65, 12.84, 15.16, 16.19, 18.01],
            5: [9.70, 9.76, 10.14, 11.02, 13.48, 15.89, 17.12, 19.19],
        },
    },
    # USPS Ground Advantage retail, Notice 123, effective July 12, 2026 (zones 1–8)
    "base_usps_retail": {
        "zones": ZONE_MILES,
        "rates": {
            1: [9.55, 9.95, 10.20, 10.60, 10.95, 11.35, 11.95, 12.90],
            2: [10.80, 11.50, 12.20, 13.00, 14.10, 15.10, 16.45, 19.05],
            3: [11.30, 12.00, 12.65, 13.70, 14.95, 16.45, 18.95, 22.40],
            4: [12.25, 12.75, 13.65, 14.85, 16.40, 18.30, 20.90, 24.25],
            5: [12.95, 13.55, 14.55, 15.80, 17.45, 19.60, 22.40, 26.05],
        },
    },
    # UPS Ground published daily rates, effective Dec 22, 2025 (zones 2–8, no zone 1)
    "high_ups_ground": {
        "zones": ZONE_MILES[1:],
        "rates": {
            1: [11.99, 12.38, 13.51, 14.10, 14.60, 14.83, 15.03],
            2: [12.85, 14.15, 15.46, 15.79, 16.49, 17.18, 17.40],
            3: [13.38, 14.91, 16.08, 16.93, 17.65, 18.28, 19.14],
            5: [14.19, 15.71, 17.15, 18.65, 19.45, 20.54, 21.72],
        },
    },
}

def calibrate(card):
    """Fit price = a + b*weight + c*miles + e*weight*miles to a carrier rate card."""
    rows = [(w, d, p) for w, prices in card["rates"].items() for d, p in zip(card["zones"], prices)]
    t = pd.DataFrame(rows, columns=["weight_lb", "miles", "price"])
    X = np.column_stack([np.ones(len(t)), t["weight_lb"], t["miles"], t["weight_lb"] * t["miles"]])
    coef, *_ = np.linalg.lstsq(X, t["price"], rcond=None)
    pred = X @ coef
    r2 = 1 - ((t["price"] - pred) ** 2).sum() / ((t["price"] - t["price"].mean()) ** 2).sum()
    return {"a": coef[0], "b": coef[1], "c": coef[2], "e": coef[3],
            "r2": r2, "max_error_usd": (t["price"] - pred).abs().max()}

formulas = pd.DataFrame({name: calibrate(card) for name, card in RATE_CARDS.items()}).T
display(formulas.round(5))

formulas.to_csv("../data/processed/cost_formulas.csv", index_label="variant")

,a,b,c,e,r2,max_error_usd
low_usps_commercial,7.02161,0.46895,0.00084,0.00086,0.98256,1.11336
base_usps_retail,9.23454,0.72936,0.00098,0.00112,0.97003,1.87662
high_ups_ground,12.18854,0.60382,0.00102,0.00049,0.92457,1.36739


## Orders as parcels
Each order comes from exactly one factory, so one order = one parcel. Weight = units × 2 oz + 0.5 lb packaging, rounded up to the next full pound.

In [2]:
P = "../data/processed/"
sales = pd.read_csv(P + "sales.csv", dtype={"postal_code": str})
products = pd.read_csv(P + "products.csv")
factories = pd.read_csv(P + "factories.csv")
zips = pd.read_csv(P + "zips.csv", dtype={"zip": str})
road = pd.read_csv(P + "road_distances.csv", dtype={"zip": str})

lines = sales.merge(products[["product_id", "factory"]], on="product_id")
orders = lines.groupby("order_id").agg(
    postal_code=("postal_code", "first"),
    current_factory=("factory", "first"),
    divisions=("division", lambda x: frozenset(x)),
    units=("units", "sum"),
    sales=("sales", "sum"),
)
orders["weight_lb"] = orders["units"] * 2 / 16 + 0.5
orders["billable_lb"] = np.ceil(orders["weight_lb"])

print("Parcels:", len(orders))
print("Billable weight distribution:", orders["billable_lb"].value_counts().sort_index().to_dict())

Parcels: 8389
Billable weight distribution: {1.0: 5045, 2.0: 3128, 3.0: 198, 4.0: 17, 5.0: 1}


## Scenarios
- **current** – each product ships from its assigned factory
- **realistic** – nearest factory (by road) among factories that already make products of the same division
- **optimum** – nearest factory (by road) among all five factories (theoretical: assumes any factory can make any product)

In [3]:
def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))

# Distance tables: rows = ZIP codes, columns = factories
road_wide = road.pivot(index="zip", columns="factory", values="road_miles")
pairs = zips.merge(factories, how="cross")
pairs["straight_miles"] = haversine_miles(pairs["lat"], pairs["lng"], pairs["latitude"], pairs["longitude"])
straight_wide = pairs.pivot(index="zip", columns="factory", values="straight_miles")

division_factories = products.groupby("division")["factory"].apply(set).to_dict()
all_factories = set(factories["factory"])

def nearest_factory(postal_code, candidates):
    return road_wide.loc[postal_code, sorted(candidates)].idxmin()

orders["optimum_factory"] = [nearest_factory(z, all_factories) for z in orders["postal_code"]]
orders["realistic_factory"] = [
    nearest_factory(z, set.intersection(*(division_factories[d] for d in divs)))
    for z, divs in zip(orders["postal_code"], orders["divisions"])
]

SCENARIOS = ["current", "realistic", "optimum"]
for sc in SCENARIOS:
    fac = orders[f"{sc}_factory"]
    orders[f"{sc}_road_miles"] = [road_wide.at[z, f] for z, f in zip(orders["postal_code"], fac)]
    orders[f"{sc}_straight_miles"] = [straight_wide.at[z, f] for z, f in zip(orders["postal_code"], fac)]

assert (orders["realistic_road_miles"] <= orders["current_road_miles"] + 1e-9).all()
assert (orders["optimum_road_miles"] <= orders["realistic_road_miles"] + 1e-9).all()

print("Orders reassigned – realistic:", (orders["realistic_factory"] != orders["current_factory"]).sum(),
      "| optimum:", (orders["optimum_factory"] != orders["current_factory"]).sum())

Orders reassigned – realistic: 4412 | optimum: 5949


## Shipping cost and savings
Cost per parcel is calculated for each scenario and each of the three price variants. Savings are shown relative to the current model.

In [4]:
def shipping_cost(weight, miles, f):
    return f["a"] + f["b"] * weight + f["c"] * miles + f["e"] * weight * miles

for variant, f in formulas.iterrows():
    for sc in SCENARIOS:
        orders[f"cost_{sc}_{variant}"] = shipping_cost(orders["billable_lb"], orders[f"{sc}_straight_miles"], f)

summary = []
for sc in SCENARIOS:
    row = {"scenario": sc,
           "total_road_miles": orders[f"{sc}_road_miles"].sum(),
           "avg_road_miles": orders[f"{sc}_road_miles"].mean()}
    for variant in formulas.index:
        row[f"cost_{variant}"] = orders[f"cost_{sc}_{variant}"].sum()
    summary.append(row)
summary = pd.DataFrame(summary).set_index("scenario")
savings_pct = (1 - summary / summary.loc["current"]) * 100

display(summary.round(0))
display(savings_pct.round(1))

,total_road_miles,avg_road_miles,cost_low_usps_commercial,cost_base_usps_retail,cost_high_ups_ground
scenario,,,,,
current,12121184.0,1445.0,86063.0,113012.0,127479.0
realistic,6366539.0,759.0,75703.0,100118.0,118829.0
optimum,4922708.0,587.0,73247.0,97062.0,116776.0


,total_road_miles,avg_road_miles,cost_low_usps_commercial,cost_base_usps_retail,cost_high_ups_ground
scenario,,,,,
current,0.0,0.0,0.0,0.0,0.0
realistic,47.5,47.5,12.0,11.4,6.8
optimum,59.4,59.4,14.9,14.1,8.4


**Findings:**
- Road miles drop by **47.5%** (realistic) and **59.4%** (optimum) – average distance per parcel falls from ~1,445 to ~759 / ~587 miles
- Shipping costs drop much less: **6.8–12.0%** (realistic) and **8.4–14.9%** (optimum), depending on the price variant
- Reason: parcel prices have a large fixed per-parcel component (~$7–12) that does not depend on distance
- Savings are positive in all three price variants → the direction of the conclusion does not depend on the carrier, only its size does
- The realistic scenario captures ~80% of the theoretical optimum's benefit, using only factories that already make each division's products

In [5]:
total_sales = orders["sales"].sum()
cost_share = summary[[c for c in summary.columns if c.startswith("cost_")]] / total_sales * 100

print(f"Total sales: ${total_sales:,.0f} | Avg order value: ${orders['sales'].mean():.2f}")
display(cost_share.round(1))

Total sales: $138,830 | Avg order value: $16.55


,cost_low_usps_commercial,cost_base_usps_retail,cost_high_ups_ground
scenario,,,
current,62.0,81.4,91.8
realistic,54.5,72.1,85.6
optimum,52.8,69.9,84.1


In [6]:
print(f"Gross margin: {lines['gross_profit'].sum() / lines['sales'].sum() * 100:.1f}%")

Gross margin: 65.9%


**Shipping cost vs. order value:**
- Average order value is only ~$16.55; shipping would consume **62–92%** of sales in the current model (53–84% after optimization)
- With a gross margin of 65.9%, paying parcel rates for every order would wipe out most or all of the margin
- In practice the company would likely pass shipping costs to customers or ship consolidated – another reason the dollar values are illustrative
- Implication for recommendations: with orders this small, the number of parcels (e.g. a minimum order value or free-shipping threshold) may be a stronger cost lever than distance

In [7]:
export = orders.drop(columns=["divisions"]).reset_index()
export.to_csv("../data/processed/orders_scenarios.csv", index=False)
print("Saved:", export.shape)

Saved: (8389, 24)
